<a href="https://colab.research.google.com/github/Epot12/QML_project/blob/master/QML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#QML PROJECT

In [2]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00


In [3]:
import os
import shutil
from google.colab import drive
import torch
import torch.nn as nn
import pandas as pd
import pydicom
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import swin_b, Swin_B_Weights

drive.mount('/content/drive')

Mounted at /content/drive


##Exploratory Data Analysis INbreast

In [6]:
dataset_root = '/content/drive/MyDrive/QML/INbreast/INbreast.zip'

print(f"Path analysis: {dataset_root}")

# Trova tutti i file DICOM (.dcm) ricorsivamente
# INbreast a volte ha estensione .dcm, a volte nessuna estensione. Cerco tutto.
dicom_files = []
for root, dirs, files in os.walk(dataset_root):
    for file in files:
        # Controllo euristico: i file DICOM di solito hanno estensione .dcm o nessuna estensione
        # Ignoro file di sistema o excel per ora
        if not file.endswith(('.xls', '.xlsx', '.csv', '.xml', '.txt', '.zip')):
            dicom_files.append(os.path.join(root, file))

print(f"Trovati {len(dicom_files)} potenziali file DICOM.")

# Estrazione Metadati dai DICOM
data_list = []

print("Estrazione metadati in corso")
for file_path in dicom_files:
    try:
        # Legge solo l'header per velocità (stop_before_pixels=True)
        ds = pydicom.dcmread(file_path, stop_before_pixels=True)

        # Estraggo le feature chiave
        data_list.append({
            'file_path': file_path,
            'PatientID': ds.get('PatientID', 'Unknown'),
            'StudyDate': ds.get('StudyDate', 'Unknown'),
            'Laterality': ds.get('ImageLaterality', 'Unknown'), # L o R
            'ViewPosition': ds.get('ViewPosition', 'Unknown'),  # CC o MLO
            'Modality': ds.get('Modality', 'Unknown'),          # Dovrebbe essere MG
            'Rows': ds.get('Rows', 0),
            'Columns': ds.get('Columns', 0)
        })
    except Exception as e:
        # Se il file non è un DICOM valido viene ignorato
        continue

# Creo il DataFrame delle immagini
df_images = pd.DataFrame(data_list)

print("Creazione DataFrame Immagini completata.")
print(f"Dimensioni dataset immagini: {df_images.shape}")
display(df_images.head())

Path analysis: /content/drive/MyDrive/QML/INbreast/INbreast.zip
Trovati 0 potenziali file DICOM.
Estrazione metadati in corso
Creazione DataFrame Immagini completata.
Dimensioni dataset immagini: (0, 0)


""


Nel codice seguente si verifica se esistono degli split predefiniti. Inoltre vengono cercate le diagnosi.

In [ ]:
# Cerca il file Excel con le diagnosi (BI-RADS)
xls_files = glob.glob(os.path.join(dataset_root, '**/*.xls*'), recursive=True)
csv_files = glob.glob(os.path.join(dataset_root, '**/*.csv'), recursive=True)
metadata_files = xls_files + csv_files

print("\n--- ANALISI FEATURES E LABELS ---")
if metadata_files:
    print(f"Trovato file di metadati/label: {metadata_files[0]}")
    meta_path = metadata_files[0]

    # Carica a seconda del formato
    if meta_path.endswith('csv'):
        df_labels = pd.read_csv(meta_path, sep=';') # Spesso INbreast usa ; come separatore
    else:
        df_labels = pd.read_excel(meta_path)

    print("Prime righe del file Labels:")
    display(df_labels.head())

    print("\n Colonne disponibili nel file Labels (Features):")
    print(df_labels.columns.tolist())
else:
    print("ATTENZIONE: Nessun file Excel/CSV trovato nella cartella.")
    print("INbreast di solito ha un file 'INbreast.xls'. Senza di esso non ci sono le diagnosi (Benigno/Maligno).")

# Controllo Split (Train/Test)
# INbreast è famoso per NON avere uno split ufficiale. Verifica.
unique_patients = df_images['PatientID'].nunique()
print(f"\n--- STRUTTURA DATASET ---")
print(f"Totale Pazienti unici: {unique_patients}")
print(f"Totale Immagini: {len(df_images)}")

# Check rapido sulla distribuzione delle Viste
if not df_images.empty:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    sns.countplot(data=df_images, x='ViewPosition')
    plt.title('Distribuzione Viste (CC vs MLO)')

    plt.subplot(1, 2, 2)
    sns.countplot(data=df_images, x='Laterality')
    plt.title('Distribuzione Lateralità (L vs R)')

    plt.show()
else:
    print("Nessun dato immagine da visualizzare.")

### Swin B Transformer classico

In [ ]:
weights = Swin_B_Weights.IMAGENET1K_V1
model = swin_b(weights=weights)

In [ ]:
for param in model.parameters():
    param.requires_grad = True

num_features = model.head.in_features

In [ ]:
model.head = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.ReLU(),
    nn.Linear(1024, 16),
    nn.Linear(16, 2)
)